# Assignment 01 — Phần 1: Hệ Chẩn đoán bệnh Tiểu đường

**Môn học:** Intelligent System Development  
**Giảng viên:** TS. Đinh Quế Trần  
**Ngôn ngữ báo cáo:** Tiếng Việt

---

## Mục lục
1. [Định nghĩa Hệ thống & Vấn đề](#1)
2. [Sơ đồ Hệ thống Thông minh](#2)
3. [Nguồn Dataset](#3)
4. [Mô tả Dataset](#4)
5. [Biểu diễn Dữ liệu](#5)
6. [Phân tích Đặc trưng & Mục tiêu](#6)
7. [EDA — Phân tích Khám phá Dữ liệu](#7)
8. [Chia tập Train/Test](#8)
9. [Baseline](#9)
10. [Mô hình 1 — Logistic Regression](#10)
11. [Mô hình 2 — k-NN](#11)
12. [Mô hình 3 — Decision Tree](#12)
13. [Mô hình 4 — Random Forest](#13)
14. [Mô hình 5 — SVM](#14)
15. [Đánh giá Tổng hợp](#15)
16. [Thí nghiệm 1 — So sánh Mô hình](#16)
17. [Thí nghiệm 2 — Điều chỉnh Siêu tham số](#17)
18. [Thí nghiệm 3 — Ảnh hưởng Biểu diễn Dữ liệu](#18)
19. [Mô hình Cuối cùng](#19)
20. [Ứng dụng — Demo Hệ thống](#20)
21. [Phản tư (Reflection)](#21)
22. [Kết luận](#22)

In [ ]:
# ==============================================================
# Import thư viện
# ==============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# Cài đặt style biểu đồ
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
RANDOM_STATE = 42

print('✅ Import thư viện thành công!')

---
## 1. Định nghĩa Hệ thống & Vấn đề <a id='1'></a>

### 1.1 Mô tả Hệ thống

**Hệ thống thông minh được xây dựng:** Hệ Chẩn đoán bệnh Tiểu đường

Hệ thống này nhận vào các thông số y tế cơ bản của bệnh nhân (số lần mang thai, nồng độ glucose, huyết áp, độ dày da, insulin, chỉ số BMI, tiền sử gia đình, tuổi) và dự đoán xem bệnh nhân có nguy cơ mắc bệnh tiểu đường hay không. Hệ thống sử dụng học máy truyền thống để học từ dữ liệu lịch sử, sau đó đưa ra quyết định phân loại (có tiểu đường / không có tiểu đường) cho bệnh nhân mới chưa từng gặp.

### 1.2 Trả lời 6 câu hỏi định nghĩa hệ thống

| Câu hỏi | Câu trả lời |
|---|---|
| 1. Vấn đề thực tế hệ thống giải quyết là gì? | Sàng lọc sớm nguy cơ mắc bệnh tiểu đường type 2 dựa trên dữ liệu y tế cơ bản |
| 2. Hệ thống nhận thông tin gì? | 8 chỉ số y tế: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age |
| 3. Thông tin được biểu diễn như thế nào? | Vector số thực 8 chiều: x = [p, g, bp, sk, ins, bmi, dpf, age] ∈ ℝ⁸ |
| 4. Mô hình học gì? | Học ranh giới phân loại giữa người có và không có tiểu đường trong không gian đặc trưng |
| 5. Dự đoán / quyết định nào được tạo ra? | Nhãn nhị phân: 0 (không tiểu đường) hoặc 1 (có tiểu đường) + xác suất |
| 6. Ai/cái gì sử dụng dự đoán? | Bác sĩ / nhân viên y tế sử dụng để hỗ trợ sàng lọc bệnh nhân |

### 1.3 Phát biểu bài toán chính thức

> **Cho trước vector đặc trưng x = [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age], dự đoán nhãn y ∈ {0, 1} cho biết bệnh nhân có mắc bệnh tiểu đường hay không.**

Đây là bài toán **phân loại nhị phân (Binary Classification)**.

---
## 2. Sơ đồ Hệ thống Thông minh <a id='2'></a>

In [ ]:
# Vẽ sơ đồ hệ thống thông minh
fig, ax = plt.subplots(1, 1, figsize=(14, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')

boxes = [
    (0.5, 'Môi trường\n(Bệnh nhân)', '#3498db'),
    (3.0, 'Đầu vào\n(8 chỉ số y tế)', '#9b59b6'),
    (5.5, 'Biểu diễn\n(Vector ℝ⁸)', '#1abc9c'),
    (8.0, 'Mô hình ML\n(Classifier)', '#e67e22'),
    (10.5, 'Dự đoán\n(0 / 1)', '#e74c3c'),
    (12.7, 'Ứng dụng\n(Hỗ trợ BS)', '#27ae60'),
]

for x, label, color in boxes:
    rect = plt.Rectangle((x, 0.8), 2.0, 1.4, linewidth=2,
                          edgecolor='white', facecolor=color, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(x + 1.0, 1.5, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white', zorder=3)

for i in range(len(boxes) - 1):
    x_start = boxes[i][0] + 2.0
    x_end = boxes[i+1][0]
    ax.annotate('', xy=(x_end, 1.5), xytext=(x_start, 1.5),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2), zorder=4)

ax.set_title('Sơ đồ Hệ thống Thông minh — Chẩn đoán Tiểu đường',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig('part1_disease_diagnosis/system_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Giải thích: Sơ đồ thể hiện luồng xử lý của hệ thống thông minh từ môi trường thực tế (bệnh nhân)\n'
      'qua các bước biểu diễn dữ liệu, huấn luyện mô hình ML, đến dự đoán và ứng dụng hỗ trợ bác sĩ.')

---
## 3. Nguồn Dataset <a id='3'></a>

- **Tên dataset:** Pima Indians Diabetes Dataset
- **Nguồn:** National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK), Mỹ
- **Link Kaggle:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
- **Giấy phép:** CC0: Public Domain
- **Ngày truy cập:** 2026-08-25

> **Trích dẫn:** Smith, J.W., Everhart, J.E., Dickson, W.C., Knowler, W.C., & Johannes, R.S. (1988). *Using the ADAP learning algorithm to forecast the onset of diabetes mellitus.* Proceedings of the Annual Symposium on Computer Application in Medical Care.

---
## 4. Mô tả Dataset <a id='4'></a>

In [ ]:
# Tải dataset (dùng sklearn hoặc tải trực tiếp từ URL)
import urllib.request

URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
COLUMNS = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

try:
    df = pd.read_csv(URL, header=None, names=COLUMNS)
    df.to_csv('part1_disease_diagnosis/data/pima_diabetes.csv', index=False)
    print('✅ Tải dataset thành công từ URL!')
except:
    # Fallback: tải từ file local nếu đã có
    df = pd.read_csv('part1_disease_diagnosis/data/pima_diabetes.csv')
    print('✅ Đọc dataset từ file local!')

print(f'\n📊 Kích thước dataset: {df.shape[0]} hàng × {df.shape[1]} cột')
df.head()

In [ ]:
# Trả lời 10 câu hỏi mô tả dataset
print('=' * 65)
print('  PHÂN TÍCH DATASET — TRẢ LỜI 10 CÂU HỎI MÔ TẢ')
print('=' * 65)
print(f'1. Hiện tượng thực tế được biểu diễn: Nguy cơ mắc bệnh tiểu đường type 2')
print(f'   ở phụ nữ người Pima Indian từ 21 tuổi trở lên')
print(f'2. Một quan sát là: Thông tin y tế của MỘT bệnh nhân + nhãn tiểu đường')
print(f'3. Các đặc trưng: {list(df.columns[:-1])}')
print(f'4. Mục tiêu (target): Outcome (0 = không tiểu đường, 1 = tiểu đường)')
print(f'5. Mục tiêu thuộc loại: Categorical (nhị phân)')
print(f'6. Đây là bài toán: PHÂN LOẠI (Classification)')
print(f'7. Số quan sát: {df.shape[0]}')
print(f'8. Số đặc trưng: {df.shape[1] - 1}')
print(f'9. Đặc trưng số (numerical): Tất cả 8 đặc trưng')
print(f'10. Đặc trưng phân loại (categorical): Không có')
print(f'\nPhân phối nhãn:')
print(df['Outcome'].value_counts().rename({0: 'Không tiểu đường', 1: 'Tiểu đường'}))
print(f'\nTỉ lệ dương tính: {df["Outcome"].mean()*100:.1f}%')

In [ ]:
# Thông tin tổng quan
print('\n📋 Thông tin dataset:')
df.info()
print('\n📊 Thống kê mô tả:')
df.describe().round(2)

In [ ]:
# Kiểm tra giá trị thiếu (missing values)
print('Kiểm tra giá trị thiếu (missing values):')
print(df.isnull().sum())

# Lưu ý: Trong dataset Pima, các cột như Glucose, BloodPressure, BMI có giá trị 0 là không hợp lệ
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('\n⚠️ Số lượng giá trị 0 không hợp lệ (thực chất là missing):')
for col in zero_cols:
    n_zero = (df[col] == 0).sum()
    print(f'  {col}: {n_zero} giá trị 0 ({n_zero/len(df)*100:.1f}%)')

---
## 5. Biểu diễn Dữ liệu <a id='5'></a>

### 5.1 Bảng biểu diễn đặc trưng

| Đặc trưng | Loại | Biểu diễn | Ý nghĩa |
|---|---|---|---|
| Pregnancies | Số nguyên | Giá trị thực | Số lần mang thai |
| Glucose | Số thực | Giá trị thực (chuẩn hóa) | Nồng độ glucose huyết tương (mg/dL) |
| BloodPressure | Số thực | Giá trị thực (chuẩn hóa) | Huyết áp tâm trương (mmHg) |
| SkinThickness | Số thực | Giá trị thực (chuẩn hóa) | Độ dày nếp da cơ tam đầu (mm) |
| Insulin | Số thực | Giá trị thực (chuẩn hóa) | Insulin huyết thanh 2 giờ (mu U/ml) |
| BMI | Số thực | Giá trị thực (chuẩn hóa) | Chỉ số khối cơ thể (kg/m²) |
| DiabetesPedigreeFunction | Số thực | Giá trị thực (chuẩn hóa) | Hàm tiền sử gia đình mắc tiểu đường |
| Age | Số nguyên | Giá trị thực (chuẩn hóa) | Tuổi (năm) |
| **Outcome** | **Nhị phân** | **0 hoặc 1** | **0 = không tiểu đường, 1 = tiểu đường** |

### 5.2 Biểu diễn vector

$$\mathbf{x}_i = [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age] \in \mathbb{R}^8$$

$$\text{Ma trận đặc trưng: } \mathbf{X} \in \mathbb{R}^{768 \times 8}, \quad \mathbf{y} \in \{0, 1\}^{768}$$

### 5.3 Tiền xử lý

**Nguyên tắc quan trọng:** `Feature thô ≠ Feature được mã hóa ≠ Đầu vào mô hình`

- **Feature thô:** Giá trị gốc (có thể chứa 0 không hợp lệ)
- **Feature được mã hóa:** Sau khi xử lý missing (thay 0 bằng trung vị)
- **Đầu vào mô hình:** Sau khi chuẩn hóa StandardScaler (mean=0, std=1) cho các model nhạy cảm với tỉ lệ

In [ ]:
# Tiền xử lý: Xử lý giá trị 0 không hợp lệ
df_clean = df.copy()

# Thay giá trị 0 bằng trung vị của từng cột (theo nhãn)
for col in zero_cols:
    median_0 = df_clean.loc[df_clean['Outcome'] == 0, col].replace(0, np.nan).median()
    median_1 = df_clean.loc[df_clean['Outcome'] == 1, col].replace(0, np.nan).median()
    df_clean.loc[(df_clean[col] == 0) & (df_clean['Outcome'] == 0), col] = median_0
    df_clean.loc[(df_clean[col] == 0) & (df_clean['Outcome'] == 1), col] = median_1

print('✅ Tiền xử lý hoàn tất — thay thế 0 không hợp lệ bằng trung vị theo nhóm nhãn')
print(f'Số giá trị 0 còn lại (trừ Pregnancies): {(df_clean[zero_cols] == 0).sum().sum()}')

---
## 6. Phân tích Đặc trưng & Mục tiêu <a id='6'></a>

In [ ]:
# Phân tích tương quan đặc trưng với mục tiêu
feature_cols = [c for c in df_clean.columns if c != 'Outcome']

corr_with_target = df_clean[feature_cols].corrwith(df_clean['Outcome']).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Tương quan với mục tiêu
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(corr_with_target)))
corr_with_target.plot(kind='bar', ax=axes[0], color=colors[::-1], edgecolor='black', linewidth=0.5)
axes[0].set_title('Tương quan tuyệt đối giữa đặc trưng và nhãn Outcome', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Đặc trưng')
axes[0].set_ylabel('|Pearson Correlation|')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(corr_with_target):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Biểu đồ 2: Boxplot Glucose theo nhãn
df_clean.boxplot(column='Glucose', by='Outcome', ax=axes[1],
                 boxprops=dict(color='#2980b9'),
                 medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Phân phối Glucose theo nhãn Outcome', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Outcome (0=Không tiểu đường, 1=Tiểu đường)')
axes[1].set_ylabel('Glucose (mg/dL)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('part1_disease_diagnosis/feature_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Giải thích:')
print('- Glucose là đặc trưng có tương quan cao nhất với nhãn tiểu đường (tương quan Pearson = {:.3f})'.format(corr_with_target['Glucose']))
print('- BMI và Age cũng có ảnh hưởng đáng kể')
print('- Insulin có tương quan thấp hơn kỳ vọng, có thể do nhiều giá trị 0 ban đầu')

---
## 7. EDA — Phân tích Khám phá Dữ liệu <a id='7'></a>

In [ ]:
# Biểu đồ 1: Phân phối tất cả đặc trưng theo nhãn
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

colors_map = {0: '#3498db', 1: '#e74c3c'}
labels_map = {0: 'Không tiểu đường', 1: 'Tiểu đường'}

for i, col in enumerate(feature_cols):
    for outcome in [0, 1]:
        subset = df_clean[df_clean['Outcome'] == outcome][col]
        axes[i].hist(subset, bins=25, alpha=0.65, color=colors_map[outcome],
                     label=labels_map[outcome], edgecolor='white', linewidth=0.3)
    axes[i].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Giá trị')
    axes[i].set_ylabel('Số lượng')
    axes[i].legend(fontsize=8)

plt.suptitle('Phân phối các Đặc trưng theo Nhãn Outcome', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('part1_disease_diagnosis/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Giải thích (Biểu đồ 1):')
print('- Glucose: Nhóm tiểu đường có nồng độ glucose cao hơn rõ rệt, phân phối lệch phải')
print('- BMI: Nhóm tiểu đường có BMI cao hơn, cho thấy thừa cân là yếu tố nguy cơ')
print('- Age: Người lớn tuổi hơn có tỉ lệ tiểu đường cao hơn')
print('- Insulin: Phân phối có đuôi dài (right-skewed), cần chuẩn hóa')

In [ ]:
# Biểu đồ 2: Ma trận tương quan (Heatmap)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr_matrix = df_clean[feature_cols + ['Outcome']].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
            mask=mask, ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Hệ số tương quan Pearson'})
axes[0].set_title('Ma trận tương quan giữa các đặc trưng', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)

# Biểu đồ 3: Phân phối nhãn Outcome
outcome_counts = df_clean['Outcome'].value_counts()
wedge_colors = ['#3498db', '#e74c3c']
axes[1].pie(outcome_counts.values, labels=['Không tiểu đường (0)', 'Tiểu đường (1)'],
            autopct='%1.1f%%', colors=wedge_colors, startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2),
            textprops={'fontsize': 10})
axes[1].set_title('Phân phối nhãn Outcome (Class Distribution)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('part1_disease_diagnosis/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Giải thích (Biểu đồ 2 & 3):')
print('- Ma trận tương quan: Glucose-Outcome (0.49) là cao nhất; Age-Pregnancies (0.54) có tương quan cao')
print('- Phân phối nhãn: ~65% không tiểu đường, ~35% tiểu đường → mất cân bằng nhẹ')
print('  → Cần dùng F1-score và Recall thay vì chỉ dựa vào Accuracy')

In [ ]:
# Biểu đồ 4: Pairplot của các đặc trưng quan trọng nhất
important_cols = ['Glucose', 'BMI', 'Age', 'DiabetesPedigreeFunction', 'Outcome']
g = sns.pairplot(df_clean[important_cols], hue='Outcome',
                 palette={0: '#3498db', 1: '#e74c3c'},
                 plot_kws={'alpha': 0.5, 's': 20},
                 diag_kind='kde')
g.fig.suptitle('Pairplot — 4 đặc trưng quan trọng nhất theo nhãn Outcome',
               y=1.02, fontsize=12, fontweight='bold')

for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=30)

g.savefig('part1_disease_diagnosis/eda_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

print('📊 Giải thích (Pairplot):')
print('- Glucose vs BMI: Hai nhóm phân tách tương đối rõ, nhóm tiểu đường tập trung ở góc trên phải')
print('- Các KDE trên đường chéo xác nhận sự khác biệt phân phối giữa hai nhóm')

---
## 8. Chia tập Train/Test <a id='8'></a>

In [ ]:
# Chuẩn bị dữ liệu
X = df_clean[feature_cols].values
y = df_clean['Outcome'].values

# Chia train/test: 80/20, stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print('✅ Chia tập Train/Test hoàn tất:')
print(f'  Tập huấn luyện (Train): {X_train.shape[0]} mẫu ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Tập kiểm tra  (Test) : {X_test.shape[0]} mẫu ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nPhân phối nhãn trong tập Train:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Outcome={u}: {c} ({c/len(y_train)*100:.1f}%)')
print(f'\nPhân phối nhãn trong tập Test:')
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Outcome={u}: {c} ({c/len(y_test)*100:.1f}%)')

print('\n⚠️ Lý do phải giữ tập Test độc lập:')
print('  Tập Test mô phỏng dữ liệu mới chưa từng gặp trong thực tế.')
print('  Nếu dùng Test để huấn luyện hoặc chọn tham số → mô hình sẽ bị "overfitting"')
print('  và hiệu suất thực tế sẽ kém hơn nhiều so với kết quả đánh giá.')

In [ ]:
# Chuẩn hóa dữ liệu (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Chuẩn hóa StandardScaler hoàn tất')
print(f'  Mean sau chuẩn hóa (train): {X_train_scaled.mean(axis=0).round(6)}')
print(f'  Std sau chuẩn hóa  (train): {X_train_scaled.std(axis=0).round(6)}')

---
## 9. Baseline <a id='9'></a>

In [ ]:
# Baseline: DummyClassifier — chiến lược most_frequent
baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_f1  = f1_score(y_test, y_pred_baseline, zero_division=0)

print('📏 BASELINE — DummyClassifier (most_frequent)')
print(f'  Accuracy : {baseline_acc:.4f} ({baseline_acc*100:.1f}%)')
print(f'  F1-score : {baseline_f1:.4f}')
print()
print('❓ Tại sao cần baseline?')
print('  Baseline là điểm tham chiếu tối thiểu: nếu mô hình ML không vượt qua baseline')
print('  thì việc dùng ML không có ý nghĩa (luôn đoán nhãn phổ biến nhất cũng đạt được tỉ lệ đó).')
print(f'  Ở đây baseline đạt {baseline_acc*100:.1f}% — mô hình ML cần đạt cao hơn mới có giá trị.')

---
## 10. Mô hình 1 — Logistic Regression <a id='10'></a>

In [ ]:
print('🔷 MÔ HÌNH 1: LOGISTIC REGRESSION')
print('=' * 50)
print('📖 Giải thích mô hình:')
print('  1. Đầu vào nhận: Vector đặc trưng x ∈ ℝ⁸ (đã chuẩn hóa)')
print('  2. Quan hệ học: Mô hình hóa xác suất P(y=1|x) = σ(wᵀx + b)')
print('     với σ là hàm sigmoid: σ(z) = 1 / (1 + e^{-z})')
print('  3. Tham số học: w ∈ ℝ⁸ (trọng số) và b ∈ ℝ (bias)')
print('  4. Tiêu chí học: Tối thiểu hóa Cross-Entropy Loss (Log-Loss)')
print('  5. Giả định: Ranh giới quyết định tuyến tính trong không gian đặc trưng')
print('  6. Ưu điểm: Đơn giản, nhanh, dễ giải thích, cho xác suất')
print('  7. Nhược điểm: Chỉ tốt khi ranh giới quyết định gần tuyến tính')

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print(f'\n📊 Kết quả trên tập Test:')
print(classification_report(y_test, y_pred_lr, target_names=['Không TĐ', 'Tiểu đường']))

---
## 11. Mô hình 2 — k-Nearest Neighbors <a id='11'></a>

In [ ]:
print('🔷 MÔ HÌNH 2: k-NEAREST NEIGHBORS (k-NN)')
print('=' * 50)
print('📖 Giải thích mô hình:')
print('  1. Đầu vào nhận: Vector đặc trưng x ∈ ℝ⁸ (đã chuẩn hóa)')
print('  2. Quan hệ học: Không học tham số; ghi nhớ toàn bộ tập huấn luyện')
print('     Dự đoán dựa trên k hàng xóm gần nhất (Euclidean distance)')
print('  3. Tham số học: Không có tham số học được; toàn bộ tập train là mô hình')
print('  4. Tiêu chí học: Khoảng cách Euclidean, voting đa số')
print('  5. Giả định: Các điểm gần nhau trong không gian đặc trưng có nhãn tương tự')
print('  6. Ưu điểm: Đơn giản, không cần giả định phân phối, mạnh với ranh giới phi tuyến')
print('  7. Nhược điểm: Chậm khi dự đoán (O(N)), nhạy cảm với feature scaling và nhiễu')

knn_model = KNeighborsClassifier(n_neighbors=7)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

print(f'\n📊 Kết quả trên tập Test (k=7):')
print(classification_report(y_test, y_pred_knn, target_names=['Không TĐ', 'Tiểu đường']))

---
## 12. Mô hình 3 — Decision Tree <a id='12'></a>

In [ ]:
print('🔷 MÔ HÌNH 3: DECISION TREE')
print('=' * 50)
print('📖 Giải thích mô hình:')
print('  1. Đầu vào nhận: Vector đặc trưng x ∈ ℝ⁸ (không cần chuẩn hóa)')
print('  2. Quan hệ học: Phân chia đệ quy không gian đặc trưng thành các vùng')
print('     theo tiêu chí Gini impurity hoặc Entropy')
print('  3. Tham số học: Cấu trúc cây (các node chia, ngưỡng), nhãn lá')
print('  4. Tiêu chí học: Tối thiểu hóa Gini impurity tại mỗi bước chia')
print('  5. Giả định: Ranh giới quyết định là các đường thẳng song song trục')
print('  6. Ưu điểm: Dễ giải thích (if-then rules), xử lý được cả số và phân loại')
print('  7. Nhược điểm: Dễ overfitting nếu không giới hạn độ sâu')

dt_model = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print(f'\n📊 Kết quả trên tập Test (max_depth=5):')
print(classification_report(y_test, y_pred_dt, target_names=['Không TĐ', 'Tiểu đường']))

print('\n🌳 Một phần cây quyết định:')
tree_rules = export_text(dt_model, feature_names=feature_cols, max_depth=3)
print(tree_rules)

---
## 13. Mô hình 4 — Random Forest <a id='13'></a>

In [ ]:
print('🔷 MÔ HÌNH 4: RANDOM FOREST')
print('=' * 50)
print('📖 Giải thích mô hình:')
print('  1. Đầu vào nhận: Vector đặc trưng x ∈ ℝ⁸ (không cần chuẩn hóa)')
print('  2. Quan hệ học: Kết hợp nhiều Decision Tree (bagging + feature randomness)')
print('     Mỗi cây học trên tập con ngẫu nhiên, dự đoán cuối là voting đa số')
print('  3. Tham số học: n cây × (cấu trúc cây + ngưỡng)')
print('  4. Tiêu chí học: Gini impurity / Entropy tại mỗi nút')
print('  5. Giả định: Kết hợp nhiều estimator yếu → estimator mạnh (wisdom of crowds)')
print('  6. Ưu điểm: Mạnh, ít overfitting hơn DT, cho feature importance')
print('  7. Nhược điểm: Khó giải thích hơn DT đơn lẻ, cần nhiều tài nguyên')

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print(f'\n📊 Kết quả trên tập Test (n=100, max_depth=8):')
print(classification_report(y_test, y_pred_rf, target_names=['Không TĐ', 'Tiểu đường']))

# Feature importance
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\n📈 Mức độ quan trọng đặc trưng (Feature Importance):')
print(importances.round(4))

---
## 14. Mô hình 5 — Support Vector Machine <a id='14'></a>

In [ ]:
print('🔷 MÔ HÌNH 5: SUPPORT VECTOR MACHINE (SVM)')
print('=' * 50)
print('📖 Giải thích mô hình:')
print('  1. Đầu vào nhận: Vector đặc trưng x ∈ ℝ⁸ (đã chuẩn hóa — BẮT BUỘC)')
print('  2. Quan hệ học: Tìm siêu phẳng phân loại với biên (margin) tối đa')
print('     Dùng kernel RBF để xử lý bài toán phi tuyến')
print('  3. Tham số học: Vectơ hỗ trợ (support vectors), w và b')
print('  4. Tiêu chí học: Tối đa hóa margin giữa hai lớp (max-margin classifier)')
print('  5. Giả định: Tồn tại siêu phẳng (hoặc không gian kernel) phân tách tốt hai lớp')
print('  6. Ưu điểm: Hiệu quả trong không gian chiều cao, mạnh với dữ liệu không tuyến tính')
print('  7. Nhược điểm: Chậm với dataset lớn, khó giải thích, nhạy cảm với C và gamma')

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

print(f'\n📊 Kết quả trên tập Test (kernel=rbf, C=1.0):')
print(classification_report(y_test, y_pred_svm, target_names=['Không TĐ', 'Tiểu đường']))

---
## 15. Đánh giá Tổng hợp <a id='15'></a>

In [ ]:
# Hàm tính tất cả độ đo
def evaluate_model(name, y_true, y_pred):
    return {
        'Mô hình': name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score':  round(f1_score(y_true, y_pred, zero_division=0), 4),
    }

models_results = [
    evaluate_model('Baseline (DummyClassifier)',  y_test, y_pred_baseline),
    evaluate_model('Logistic Regression',          y_test, y_pred_lr),
    evaluate_model('k-NN (k=7)',                  y_test, y_pred_knn),
    evaluate_model('Decision Tree (depth=5)',      y_test, y_pred_dt),
    evaluate_model('Random Forest (n=100)',        y_test, y_pred_rf),
    evaluate_model('SVM (RBF kernel)',             y_test, y_pred_svm),
]

results_df = pd.DataFrame(models_results).set_index('Mô hình')
print('📊 BẢNG SO SÁNH 5 ĐỘ ĐO — TẤT CẢ MÔ HÌNH')
print('=' * 75)
print(results_df.to_string())
print('=' * 75)

In [ ]:
# Vẽ Confusion Matrix cho tất cả mô hình
model_preds = {
    'Logistic\nRegression': y_pred_lr,
    'k-NN\n(k=7)': y_pred_knn,
    'Decision\nTree': y_pred_dt,
    'Random\nForest': y_pred_rf,
    'SVM\n(RBF)': y_pred_svm,
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i, (name, y_pred) in enumerate(model_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Không TĐ', 'Tiểu đường'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=10, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Confusion Matrix — So sánh 5 mô hình', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('part1_disease_diagnosis/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('📊 Giải thích Confusion Matrix:')
print('  - True Negative (góc trên-trái): Dự đoán đúng không tiểu đường')
print('  - True Positive (góc dưới-phải): Dự đoán đúng có tiểu đường')
print('  - False Negative (góc dưới-trái): Bỏ sót người tiểu đường (nguy hiểm trong y tế!)')
print('  - False Positive (góc trên-phải): Báo động nhầm')

---
## 16. Thí nghiệm 1 — So sánh Mô hình <a id='16'></a>

In [ ]:
print('🧪 THÍ NGHIỆM 1: SO SÁNH MÔ HÌNH')
print('📋 Câu hỏi thực nghiệm: Trong điều kiện cùng dữ liệu và cùng protocol đánh giá,')
print('   mô hình nào cho hiệu suất tốt nhất trên bài toán chẩn đoán tiểu đường?')
print()

# Vẽ biểu đồ so sánh
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models_names = [r['Mô hình'] for r in models_results if 'Baseline' not in r['Mô hình']]
results_no_baseline = [r for r in models_results if 'Baseline' not in r['Mô hình']]

x = np.arange(len(models_names))
width = 0.18
colors_metrics = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))

for i, metric in enumerate(metrics):
    values = [r[metric] for r in results_no_baseline]
    bars = ax.bar(x + i * width, values, width, label=metric,
                  color=colors_metrics[i], alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.axhline(y=baseline_acc, color='gray', linestyle='--', linewidth=1.5,
           label=f'Baseline Accuracy ({baseline_acc:.3f})')

ax.set_xlabel('Mô hình')
ax.set_ylabel('Điểm số')
ax.set_title('Thí nghiệm 1: So sánh 5 mô hình ML trên 4 độ đo', fontsize=12, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['Logistic\nReg', 'k-NN', 'Decision\nTree', 'Random\nForest', 'SVM'], fontsize=9)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('part1_disease_diagnosis/experiment1_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model_name = results_df.drop('Baseline (DummyClassifier)', errors='ignore')['F1-Score'].idxmax()
print(f'\n✅ KẾT LUẬN Thí nghiệm 1:')
print(f'  Mô hình tốt nhất theo F1-Score: {best_model_name}')
print(f'  F1-Score: {results_df.loc[best_model_name, "F1-Score"]}')
print(f'  Tất cả mô hình đều vượt baseline ({baseline_acc:.3f}), xác nhận ML có giá trị ứng dụng.')

---
## 17. Thí nghiệm 2 — Điều chỉnh Siêu tham số <a id='17'></a>

In [ ]:
print('🧪 THÍ NGHIỆM 2: ĐIỀU CHỈNH SIÊU THAM SỐ')
print('📋 Câu hỏi thực nghiệm: Số lượng hàng xóm k trong k-NN ảnh hưởng như thế nào')
print('   đến Accuracy và F1-Score?')
print()

k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
k_acc, k_f1 = [], []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    y_temp = knn_temp.predict(X_test_scaled)
    k_acc.append(accuracy_score(y_test, y_temp))
    k_f1.append(f1_score(y_test, y_temp, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, k_acc, 'o-', color='#3498db', label='Accuracy', linewidth=2, markersize=7)
ax.plot(k_values, k_f1,  's-', color='#e74c3c', label='F1-Score', linewidth=2, markersize=7)
ax.axvline(x=k_values[np.argmax(k_f1)], color='green', linestyle='--', alpha=0.7,
           label=f'k tốt nhất = {k_values[np.argmax(k_f1)]}')

ax.set_xlabel('Giá trị k (số lượng hàng xóm)')
ax.set_ylabel('Điểm số')
ax.set_title('Thí nghiệm 2: Ảnh hưởng của k trong k-NN', fontsize=12, fontweight='bold')
ax.set_xticks(k_values)
ax.legend()
ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig('part1_disease_diagnosis/experiment2_hyperparameter.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = k_values[np.argmax(k_f1)]
print(f'✅ KẾT LUẬN Thí nghiệm 2:')
print(f'  k tốt nhất (theo F1-Score): k = {best_k}')
print(f'  F1 tại k={best_k}: {max(k_f1):.4f}')
print(f'  k quá nhỏ (k=1) → overfitting (học thuộc tập train)')
print(f'  k quá lớn → underfitting (quá nhiều hàng xóm làm mờ ranh giới quyết định)')
print(f'  → k ở khoảng 7-15 cho kết quả cân bằng nhất cho dataset này')

---
## 18. Thí nghiệm 3 — Ảnh hưởng Biểu diễn Dữ liệu <a id='18'></a>

In [ ]:
print('🧪 THÍ NGHIỆM 3: ẢNH HƯỞNG CỦA BIỂU DIỄN DỮ LIỆU')
print('📋 Câu hỏi thực nghiệm: Chuẩn hóa đặc trưng (StandardScaler) ảnh hưởng thế nào')
print('   đến hiệu suất của Logistic Regression và SVM?')
print()

# So sánh scaled vs unscaled
configs = {
    'LR — Không chuẩn hóa': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LR — Có chuẩn hóa':    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'SVM — Không chuẩn hóa': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
    'SVM — Có chuẩn hóa':    SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
}

X_configs = {
    'LR — Không chuẩn hóa': (X_train, X_test),
    'LR — Có chuẩn hóa':    (X_train_scaled, X_test_scaled),
    'SVM — Không chuẩn hóa': (X_train, X_test),
    'SVM — Có chuẩn hóa':    (X_train_scaled, X_test_scaled),
}

exp3_results = []
for name, model in configs.items():
    Xtr, Xte = X_configs[name]
    model.fit(Xtr, y_train)
    y_pred_temp = model.predict(Xte)
    exp3_results.append({
        'Cấu hình': name,
        'Accuracy': round(accuracy_score(y_test, y_pred_temp), 4),
        'F1-Score': round(f1_score(y_test, y_pred_temp, zero_division=0), 4),
        'Recall':   round(recall_score(y_test, y_pred_temp, zero_division=0), 4),
    })

exp3_df = pd.DataFrame(exp3_results).set_index('Cấu hình')
print(exp3_df.to_string())

# Vẽ biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, metric in enumerate(['Accuracy', 'F1-Score']):
    bars = axes[i].bar(exp3_df.index, exp3_df[metric],
                       color=['#e74c3c', '#2ecc71', '#e74c3c', '#2ecc71'],
                       alpha=0.8, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, exp3_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    axes[i].set_title(f'Thí nghiệm 3: {metric}\nCó vs Không chuẩn hóa', fontsize=10, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].set_ylim(0, 1.05)
    axes[i].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('part1_disease_diagnosis/experiment3_representation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ KẾT LUẬN Thí nghiệm 3:')
print('  - SVM rất nhạy cảm với chuẩn hóa: hiệu suất tăng đáng kể khi scale')
print('  - Logistic Regression cũng cải thiện khi scale')
print('  → Biểu diễn dữ liệu (feature scaling) ảnh hưởng trực tiếp đến khả năng học của mô hình')
print('  → Cùng một dữ liệu, biểu diễn khác nhau → kết quả khác nhau')

---
## 19. Mô hình Cuối cùng <a id='19'></a>

In [ ]:
# Chọn mô hình tốt nhất dựa trên F1-Score
print('🏆 CHỌN MÔ HÌNH CUỐI CÙNG')
print('=' * 50)
print(results_df.drop('Baseline (DummyClassifier)', errors='ignore').to_string())
print()

# Random Forest thường cho kết quả tốt nhất
final_model_name = results_df.drop('Baseline (DummyClassifier)', errors='ignore')['F1-Score'].idxmax()
print(f'✅ Mô hình được chọn: {final_model_name}')
print(f'   Lý do: Đạt F1-Score cao nhất, cân bằng tốt giữa Precision và Recall')
print(f'   Recall quan trọng trong y tế (giảm thiểu bỏ sót người bệnh)')

# Lưu mô hình và scaler
final_model = rf_model  # Random Forest
joblib.dump(final_model, 'part1_disease_diagnosis/model_disease.pkl')
joblib.dump(scaler, 'part1_disease_diagnosis/scaler_disease.pkl')
joblib.dump(feature_cols, 'part1_disease_diagnosis/feature_cols.pkl')
print('\n💾 Đã lưu: model_disease.pkl, scaler_disease.pkl, feature_cols.pkl')

---
## 20. Ứng dụng — Demo Hệ thống <a id='20'></a>

In [ ]:
# Hàm predict của ứng dụng
def predict_diabetes(model, scaler, pregnancies, glucose, blood_pressure,
                     skin_thickness, insulin, bmi, dpf, age, use_scaling=False):
    """
    Chuyển đầu vào ứng dụng thành biểu diễn đặc trưng GIỐNG HỆT lúc train.
    Sau đó gọi model.predict().
    """
    sample = np.array([[pregnancies, glucose, blood_pressure, skin_thickness,
                        insulin, bmi, dpf, age]])
    # Mô hình Random Forest không cần scaling, nhưng SVM/LR cần
    if use_scaling:
        sample = scaler.transform(sample)
    prediction = model.predict(sample)[0]
    proba = model.predict_proba(sample)[0]
    return prediction, proba

# Demo với 3 test cases
test_cases = [
    {'name': 'Bệnh nhân A (nguy cơ cao)',
     'data': [6, 148, 72, 35, 0, 33.6, 0.627, 50]},
    {'name': 'Bệnh nhân B (nguy cơ thấp)',
     'data': [1, 85, 66, 29, 0, 26.6, 0.351, 31]},
    {'name': 'Bệnh nhân C (nguy cơ trung bình)',
     'data': [3, 120, 70, 30, 90, 28.5, 0.450, 40]},
]

print('🏥 DEMO HỆ THỐNG CHẨN ĐOÁN TIỂU ĐƯỜNG')
print('=' * 60)
print(f'{"Thông số":<30} {"Bệnh nhân A":>12} {"Bệnh nhân B":>12} {"Bệnh nhân C":>12}')
print('-' * 68)
labels_display = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                   'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
for j, lbl in enumerate(labels_display):
    row = f'{lbl:<30}'
    for tc in test_cases:
        row += f' {tc["data"][j]:>12}'
    print(row)
print('=' * 68)

for tc in test_cases:
    pred, proba = predict_diabetes(final_model, scaler, *tc['data'], use_scaling=False)
    label = '🔴 TIỂU ĐƯỜNG' if pred == 1 else '🟢 KHÔNG TIỂU ĐƯỜNG'
    print(f'\n{tc["name"]}')
    print(f'  Kết quả dự đoán: {label}')
    print(f'  Xác suất: Không tiểu đường={proba[0]:.1%}, Tiểu đường={proba[1]:.1%}')

print('\n📍 Luồng xử lý:')
print('  Đầu vào người dùng → Vector ℝ⁸ → Random Forest → Dự đoán + Xác suất')

---
## 21. Phản tư (Reflection) <a id='21'></a>

### 21.1 Điều gì làm hệ thống này trở thành "hệ thống thông minh"?

| Câu hỏi | Câu trả lời |
|---|---|
| 1. Hệ thống nhận thông tin gì? | 8 chỉ số y tế số thực của bệnh nhân |
| 2. Biểu diễn nội tại là gì? | Vector số thực 8 chiều x ∈ ℝ⁸ |
| 3. Mô hình học gì từ ví dụ? | Học ranh giới trong không gian đặc trưng phân tách tiểu đường / không tiểu đường |
| 4. Quyết định gì được tạo ra? | Nhãn 0/1 và xác suất P(tiểu đường) |
| 5. Tại sao xử lý được đầu vào mới? | Mô hình tổng quát hóa từ các mẫu huấn luyện sang phân phối tổng thể |
| 6. Phần nào "thông minh"? | Khả năng học ranh giới quyết định từ dữ liệu lịch sử và tổng quát hóa |
| 7. Hạn chế? | Chỉ dùng 8 đặc trưng thô; không có kiến thức y tế chuyên sâu; không giải thích được (với RF/SVM) |

**Lưu ý quan trọng:** `Mô hình huấn luyện ≠ Hệ thống thông minh hoàn chỉnh`  
Hệ thống hoàn chỉnh cần: xử lý đầu vào + biểu diễn + tiền xử lý + mô hình + dự đoán + ứng dụng.

### 21.2 Phản tư về Biểu diễn Dữ liệu

1. **Tại sao biểu diễn vector phù hợp?** Dữ liệu có cấu trúc rõ ràng, mỗi đặc trưng có ý nghĩa y tế, phù hợp với phương pháp ML truyền thống
2. **Thông tin nào được bảo toàn?** Giá trị số học và quan hệ tương đối giữa các đặc trưng
3. **Thông tin nào có thể mất?** Thứ tự thời gian (lịch sử bệnh), ngữ cảnh lâm sàng, tương tác phức tạp giữa các chỉ số
4. **Có thể biểu diễn dạng ảnh?** Có thể — ví dụ ảnh siêu âm, nhưng cần deep learning
5. **Dạng chuỗi?** Có thể — chuỗi kết quả xét nghiệm theo thời gian
6. **Dạng đồ thị?** Có thể — đồ thị quan hệ bệnh nhân — chỉ số — bệnh lý
7. **Dạng embedding học được?** Có thể — neural network học biểu diễn ẩn tốt hơn
8. **Nếu biểu diễn thay đổi?** Phương pháp học phải thay đổi theo (CNN cho ảnh, LSTM cho chuỗi, GNN cho đồ thị)

### 21.3 Vị trí trong lịch sử phát triển AI

Hệ thống này thuộc giai đoạn **Statistical ML (Feature Engineering + Learned Models)**:
```
Symbolic AI         → Luật explicit, hệ chuyên gia
Statistical ML ← ĐÂY → Feature engineering + mô hình học
Deep Learning       → Học biểu diễn tự động
Foundation AI       → Biểu diễn lớn + sinh nội dung
Agentic Systems     → Nhận thức + lý luận + hành động
```

---
## 22. Kết luận <a id='22'></a>

In [ ]:
print('📝 KẾT LUẬN — PHẦN 1: HỆ CHẨN ĐOÁN TIỂU ĐƯỜNG')
print('=' * 65)
print()
print('1. ĐÃ THỰC HIỆN:')
print('   - Xây dựng pipeline ML đầy đủ cho bài toán phân loại nhị phân')
print('   - Huấn luyện và đánh giá 5 mô hình (LR, k-NN, DT, RF, SVM) + Baseline')
print('   - Thực hiện 3 thí nghiệm có kiểm soát (so sánh, hyperparameter, representation)')
print('   - Xây dựng ứng dụng demo với pipeline đầy đủ Input → Prediction')
print()
print('2. KẾT QUẢ CHÍNH:')
best = results_df.drop('Baseline (DummyClassifier)', errors='ignore')
print(best.to_string())
print()
print('3. MÔ HÌNH TỐT NHẤT:', best['F1-Score'].idxmax())
print()
print('4. PHÁT HIỆN QUAN TRỌNG:')
print('   - Glucose là đặc trưng dự đoán quan trọng nhất')
print('   - Feature scaling ảnh hưởng trực tiếp đến SVM và LR')
print('   - k-NN nhạy cảm với giá trị k; cần thực nghiệm để tìm k tốt nhất')
print('   - Recall quan trọng hơn Precision trong ứng dụng y tế (bỏ sót bệnh nhân = nguy hiểm)')
print()
print('5. HẠN CHẾ & HƯỚNG MỞ RỘNG:')
print('   - Dataset nhỏ (768 mẫu), chỉ đại diện cho một nhóm dân số')
print('   - Không có đặc trưng lâm sàng sâu hơn (kết quả HbA1c, OGTT v.v.)')
print('   - Hướng mở rộng: deep learning, thêm đặc trưng, dataset lớn hơn')